In [2]:
import numpy as np

In [3]:
def conjugate_gradient(A: np.ndarray, b: np.ndarray, x0: np.ndarray = None, tol: float = 1e-10, max_iter: int = 1000) -> np.ndarray:
    """
    Solve the linear system Ax = b using the Conjugate Gradient method.
    Parameters:
    A : np.ndarray
        Symmetric positive-definite matrix.
    b : np.ndarray
        Right-hand side vector.
    x0 : np.ndarray, optional
        Initial guess for the solution (default is a zero vector).
    tol : float, optional
        Tolerance for convergence (default is 1e-10).
    max_iter : int, optional
        Maximum number of iterations (default is 1000).
    """

    # number of rows
    n = A.shape[0]

    if x0 is None:
        x = np.zeros(n)
    else:
        x = x0.copy()

    r = b - A @ x
    p = r.copy()
    rs_old = np.dot(r, r)

    for i in range(max_iter):
        Ap = A @ p
        alpha = rs_old / np.dot(p, Ap)

        x = x + alpha * p
        r = r - alpha * Ap
        
        rs_new = r.T @ r
        if np.sqrt(rs_new) < tol:
            print("Converged")
            break

        beta = rs_new / rs_old

        p = r + beta * p

        rs_old = rs_new

    return x   

# Example usage
if __name__ == "__main__":
    # Define a symmetric positive-definite matrix A and vector b
    A = np.array([[4, 1], [1, 3]], dtype=float)
    b = np.array([1, 2], dtype=float)

    # Solve for x in Ax = b
    import time
    start_time = time.time()
    x = conjugate_gradient(A, b)
    end_time = time.time()
    print("Time taken:", (end_time - start_time) / 60, "minutes")
    assert np.allclose(x, np.array([0.090909, 0.636364]))
    print("Solution x:", x)

Converged
Time taken: 5.38031260172526e-06 minutes
Solution x: [0.09090909 0.63636364]


In [1]:
import numpy as np

def load_matrix_from_binary(file_path: str) -> np.ndarray:
    """Load a square matrix from a binary file with the specified format.
       the function to check the dense matrix properties: symmetric, SPD, condition number, diagonal dominance.
    """
    with open("../data/matrix.bin", "rb") as f:
        n_rows = np.fromfile(f, dtype=np.int32, count=1)[0]
        n_cols = np.fromfile(f, dtype=np.int32, count=1)[0]
        assert n_rows == n_cols, f"Not square: {n_rows}x{n_cols}"
        A = np.fromfile(f, dtype=np.float64, count=n_rows * n_cols).reshape(n_rows, n_cols)

    return A

def load_matrix_from_mtx(file_path: str) -> np.ndarray:
    """Load a square matrix from a Matrix Market (.mtx) file."""
    from scipy.io import mmread
    A = mmread(file_path).toarray()
    n_rows, n_cols = A.shape
    assert n_rows == n_cols, f"Not square: {n_rows}x{n_cols}"
    return A

def check_spd_matrix_properties(A: np.ndarray):
    """Check and print properties of the matrix A."""
    print(f"Matrix shape: {A.shape}")
    print(f"Matrix:\n{A}\n")

    # Check symmetry
    is_sym = np.allclose(A, A.T, atol=1e-10)
    print(f"✓ Symmetric: {is_sym}")

    # Check SPD via eigenvalues
    eigvals = np.linalg.eigvalsh(A)
    print(f"Eigenvalues: {eigvals}")
    is_spd = np.all(eigvals > 1e-10)
    print(f"✓ Positive definite (all eigenvalues > 0): {is_spd}")

    # Condition number
    if is_spd:
        cond_num = np.max(eigvals) / np.min(eigvals)
        print(f"Condition number: {cond_num:.2e}")

    # Check diagonal dominance
    diag = np.diag(A)
    row_sums = np.sum(np.abs(A), axis=1) - np.abs(diag)
    is_dd = np.all(np.abs(diag) > row_sums)
    print(f"✓ Diagonally dominant: {is_dd}")

In [ ]:
DATA_PATH = "../data/"

matrix_bin_path = DATA_PATH + "matrix_dense.bin"
matrix_mtx_path = DATA_PATH + "matrix_csr.mtx"

